In [1]:
!pip install pyspark pandas pyarrow jupyterlab
from pyspark.sql import SparkSession


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
spark = SparkSession.builder.appName("Testapp").master("local[*]").getOrCreate()

d:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
sc = spark.sparkContext

In [15]:
test = sc.textFile("test.csv")
print('Import Done')

Import Done


In [16]:
test.collect()

['emp_id,name,dept,city,salary',
 '101,Anu,Sales,Chennai,45000',
 '102,Ravi,IT,Bengaluru,72000',
 '103,Asha,HR,Chennai,39000',
 '104,Kiran,IT,Hyderabad,68000',
 '105,Meena,Sales,Mumbai,',
 '106,Suresh,Finance,Delhi,abc',
 '107,Divya,IT,Chennai,81000',
 '108,Arjun,HR,Mumbai,42000']

In [17]:
print(test.count())
print(test.getNumPartitions())
print(test.take(3))

9
2
['emp_id,name,dept,city,salary', '101,Anu,Sales,Chennai,45000', '102,Ravi,IT,Bengaluru,72000']


In [18]:
header = test.first()
header

'emp_id,name,dept,city,salary'

In [19]:
data = test.filter(lambda row: row != header)
data.collect()

['101,Anu,Sales,Chennai,45000',
 '102,Ravi,IT,Bengaluru,72000',
 '103,Asha,HR,Chennai,39000',
 '104,Kiran,IT,Hyderabad,68000',
 '105,Meena,Sales,Mumbai,',
 '106,Suresh,Finance,Delhi,abc',
 '107,Divya,IT,Chennai,81000',
 '108,Arjun,HR,Mumbai,42000']

In [21]:
separated_data = data.map(lambda row: row.split(","))
separated_data.collect()

[['101', 'Anu', 'Sales', 'Chennai', '45000'],
 ['102', 'Ravi', 'IT', 'Bengaluru', '72000'],
 ['103', 'Asha', 'HR', 'Chennai', '39000'],
 ['104', 'Kiran', 'IT', 'Hyderabad', '68000'],
 ['105', 'Meena', 'Sales', 'Mumbai', ''],
 ['106', 'Suresh', 'Finance', 'Delhi', 'abc'],
 ['107', 'Divya', 'IT', 'Chennai', '81000'],
 ['108', 'Arjun', 'HR', 'Mumbai', '42000']]

In [25]:
bad_data = separated_data.filter(lambda x: (not(x[4].isdigit())))
print(bad_data.collect())
good_data = separated_data.filter(lambda y: (y[4].isdigit()))
good_data.collect()

[['105', 'Meena', 'Sales', 'Mumbai', ''], ['106', 'Suresh', 'Finance', 'Delhi', 'abc']]


[['101', 'Anu', 'Sales', 'Chennai', '45000'],
 ['102', 'Ravi', 'IT', 'Bengaluru', '72000'],
 ['103', 'Asha', 'HR', 'Chennai', '39000'],
 ['104', 'Kiran', 'IT', 'Hyderabad', '68000'],
 ['107', 'Divya', 'IT', 'Chennai', '81000'],
 ['108', 'Arjun', 'HR', 'Mumbai', '42000']]

In [26]:
data_change= good_data.map(lambda a : (int(a[0]),a[1],a[2],a[3],float(a[4])))
data_change.collect()

[(101, 'Anu', 'Sales', 'Chennai', 45000.0),
 (102, 'Ravi', 'IT', 'Bengaluru', 72000.0),
 (103, 'Asha', 'HR', 'Chennai', 39000.0),
 (104, 'Kiran', 'IT', 'Hyderabad', 68000.0),
 (107, 'Divya', 'IT', 'Chennai', 81000.0),
 (108, 'Arjun', 'HR', 'Mumbai', 42000.0)]

In [29]:
bonus_data = data_change.map(lambda b: (b[0],b[1],b[2],b[3],b[4],b[4]*0.2))
bonus_data.collect()

[(101, 'Anu', 'Sales', 'Chennai', 45000.0, 9000.0),
 (102, 'Ravi', 'IT', 'Bengaluru', 72000.0, 14400.0),
 (103, 'Asha', 'HR', 'Chennai', 39000.0, 7800.0),
 (104, 'Kiran', 'IT', 'Hyderabad', 68000.0, 13600.0),
 (107, 'Divya', 'IT', 'Chennai', 81000.0, 16200.0),
 (108, 'Arjun', 'HR', 'Mumbai', 42000.0, 8400.0)]

In [30]:
new_salary = bonus_data.map(lambda c: (c[0],c[1],c[2],c[3],c[4],c[5],c[4]+c[5]))
new_salary.collect()

[(101, 'Anu', 'Sales', 'Chennai', 45000.0, 9000.0, 54000.0),
 (102, 'Ravi', 'IT', 'Bengaluru', 72000.0, 14400.0, 86400.0),
 (103, 'Asha', 'HR', 'Chennai', 39000.0, 7800.0, 46800.0),
 (104, 'Kiran', 'IT', 'Hyderabad', 68000.0, 13600.0, 81600.0),
 (107, 'Divya', 'IT', 'Chennai', 81000.0, 16200.0, 97200.0),
 (108, 'Arjun', 'HR', 'Mumbai', 42000.0, 8400.0, 50400.0)]

In [32]:
dpt_group = data_change.map(lambda a: (a[2],a[4])).reduceByKey(lambda x,y: x+y)
dpt_group.collect()

[('Sales', 45000.0), ('IT', 221000.0), ('HR', 81000.0)]

In [33]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
print("done")

done


In [36]:
schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Dept", StringType(), True),
    StructField("Salary", DoubleType(), True),
    StructField("Bonus", DoubleType(), True),
    StructField("Uptated_Salary", DoubleType(), True)
])

In [37]:
df = spark.createDataFrame(new_salary, schema)
df.show()
df.printSchema()

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "D:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3622, in main
    process()
  File "D:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "D:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\util.py", line 159, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "d:\DE Assignments\PySpark\.venv\Lib\site-packages\pyspark\sql\session.py", line 1675, in prepare
    verify_func(obj)
  File "d:\DE Assignments\PySp